# Brent Flat Price Contracts — Open Interest EDA

Analyze the relative size of each Brent flat price contract.

Brent positioning comes from two independent sources:
- **ICE Futures Europe** — the primary benchmark contract (Bloomberg's CO ticker)
- **CFTC (NYMEX)** — cash-settled lookalikes and options

### ICE Europe
- ICE Brent Crude Futures and Options - ICE Futures Europe

### CFTC Flat price codes (from `docs/brent_cot_mapping.md`):
- `06765J` — BRENT FINANCIAL - NEW YORK MERCANTILE EXCHANGE
- `06765T` — BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE
- `06765Y` — BRENT AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE
- `06765X` — EUR STYLE BRENT OPTIONS - NEW YORK MERCANTILE EXCHANGE
- `06765N` — BRENT (ICE) CALENDAR SWAP - NEW YORK MERCANTILE EXCHANGE

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## Load CFTC Data

In [ ]:
df_cftc = pd.read_csv("../../downloads/cftc/disaggregated_combined.csv", low_memory=False)

BRENT_FLAT_PRICE_CODES = [
    "06765J",  # BRENT FINANCIAL - NYMEX
    "06765T",  # BRENT CRUDE OIL LAST DAY - NYMEX
    "06765Y",  # BRENT AVG PRICE OPTIONS - NYMEX
    "06765X",  # EUR STYLE BRENT OPTIONS - NYMEX
    "06765N",  # BRENT (ICE) CALENDAR SWAP - NYMEX (added per Marouen)
]

cftc = df_cftc[df_cftc["cftc_contract_market_code"].isin(BRENT_FLAT_PRICE_CODES)].copy()
cftc["report_date"] = pd.to_datetime(cftc["report_date_as_yyyy_mm_dd"])
cftc["open_interest_all"] = pd.to_numeric(cftc["open_interest_all"], errors="coerce")
cftc["source"] = "CFTC (NYMEX)"
cftc["label"] = cftc["cftc_contract_market_code"] + " — " + cftc["market_and_exchange_names"]
print(f"CFTC Brent flat price rows: {len(cftc):,}")
print(f"Codes found: {sorted(cftc['cftc_contract_market_code'].unique())}")

## Load ICE Europe Data

In [ ]:
df_ice = pd.read_csv("../../downloads/ice/ice_cot.csv", low_memory=False)

ice = df_ice[df_ice["Market_and_Exchange_Names"].str.contains("Brent Crude", case=False, na=False)].copy()
ice["report_date"] = pd.to_datetime(ice["As_of_Date_Form_MM/DD/YYYY"])
ice["open_interest_all"] = pd.to_numeric(ice["Open_Interest_All"], errors="coerce")
ice["cftc_contract_market_code"] = "ICE_BRENT"
ice["market_and_exchange_names"] = ice["Market_and_Exchange_Names"]
ice["source"] = "ICE Europe"
ice["label"] = "ICE_BRENT — " + ice["Market_and_Exchange_Names"]
print(f"ICE Brent rows: {len(ice):,}")
print(f"Contract names: {sorted(ice['Market_and_Exchange_Names'].unique())}")

## Combine All Brent Flat Price Contracts

In [ ]:
cols = ["report_date", "cftc_contract_market_code", "market_and_exchange_names", "open_interest_all", "source", "label"]
brent = pd.concat([cftc[cols], ice[cols]], ignore_index=True)
print(f"Total Brent flat price rows: {len(brent):,}")
print(f"Date range: {brent['report_date'].min().date()} to {brent['report_date'].max().date()}")

## Average Open Interest by Contract

In [ ]:
code_labels = (
    brent.groupby("cftc_contract_market_code")["market_and_exchange_names"]
    .first()
    .to_dict()
)

source_labels = (
    brent.groupby("cftc_contract_market_code")["source"]
    .first()
    .to_dict()
)

avg_oi = (
    brent.groupby("cftc_contract_market_code")["open_interest_all"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("avg_open_interest")
)
avg_oi["contract_name"] = avg_oi.index.map(code_labels)
avg_oi["source"] = avg_oi.index.map(source_labels)
avg_oi["pct_of_total"] = (avg_oi["avg_open_interest"] / avg_oi["avg_open_interest"].sum() * 100)
avg_oi["avg_open_interest"] = avg_oi["avg_open_interest"].round(0).astype(int)
avg_oi["pct_of_total"] = avg_oi["pct_of_total"].round(2)

avg_oi[["contract_name", "source", "avg_open_interest", "pct_of_total"]]

## Open Interest Share — Pie Chart

In [ ]:
fig = px.pie(
    avg_oi.reset_index(),
    values="avg_open_interest",
    names="cftc_contract_market_code",
    hover_data=["contract_name"],
    title="Brent Flat Price Contracts — Average Open Interest Share",
)
fig.update_traces(textinfo="label+percent", textposition="outside")
fig.show()

## Open Interest Over Time by Contract

In [ ]:
fig = px.line(
    brent.sort_values("report_date"),
    x="report_date",
    y="open_interest_all",
    color="cftc_contract_market_code",
    hover_data=["market_and_exchange_names"],
    title="Brent Flat Price Contracts — Open Interest Over Time",
    labels={
        "report_date": "Report Date",
        "open_interest_all": "Open Interest",
        "cftc_contract_market_code": "Contract Code",
    },
)
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.3))
fig.show()

## Open Interest Over Time — Stacked Area (Percentage)

In [ ]:
pivot = brent.pivot_table(
    index="report_date",
    columns="cftc_contract_market_code",
    values="open_interest_all",
    aggfunc="sum",
).fillna(0)

pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig = go.Figure()
for col in pct.columns:
    label = code_labels.get(col, col)
    fig.add_trace(go.Scatter(
        x=pct.index, y=pct[col],
        name=col,
        hovertext=label,
        stackgroup="one",
        groupnorm="percent",
    ))
fig.update_layout(
    title="Brent Flat Price Contracts — OI Share Over Time (%)",
    xaxis_title="Report Date",
    yaxis_title="% of Total Brent Flat Price OI",
    legend=dict(orientation="h", yanchor="bottom", y=-0.3),
)
fig.show()

## Summary Table — Latest Reporting Date

In [ ]:
latest_date = brent["report_date"].max()
latest = brent[brent["report_date"] == latest_date].copy()

summary = (
    latest.groupby("cftc_contract_market_code")
    .agg(
        contract_name=("market_and_exchange_names", "first"),
        open_interest=("open_interest_all", "sum"),
    )
    .sort_values("open_interest", ascending=False)
)
summary["pct_of_total"] = (summary["open_interest"] / summary["open_interest"].sum() * 100).round(2)

print(f"Latest report date: {latest_date.date()}")
summary